# Alstom India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** jobsearch.alstom.com

**ATS Detection:** SAP SuccessFactors / Jobs2Web portal (Selenium-based)

In [ ]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q

In [ ]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_PARAM   = "India"  # Used in URL param ?country=India; set "" for global
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")

In [ ]:
COMPANY = "Alstom"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
print("=" * 60)
print("ALSTOM INDIA JOB SCRAPER")
print("ATS: SAP SuccessFactors / Jobs2Web (jobsearch.alstom.com)")
print("=" * 60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

BASE_URL = "https://jobsearch.alstom.com"
SEARCH_PATH = "/search/jobs"


def build_search_url(country_param, startrow=0):
    """Build a paginated search URL for the Alstom jobs portal."""
    params = f"?startrow={startrow}&sortColumn=referencedate&sortDirection=desc"
    if country_param:
        params += f"&country={country_param}"
    return BASE_URL + SEARCH_PATH + params


def parse_job_cards(soup):
    """Extract job records from a BeautifulSoup page of Alstom search results."""
    jobs = []

    # Jobs2Web renders each job inside a <tr> with class 'data-row' or similar,
    # or inside <li> / <article> elements.  Try broad selectors and fall back.
    cards = (
        soup.select("tr.data-row")
        or soup.select("[class*='jobResultItem']")  
        or soup.select("[class*='job-result']")      
        or soup.select("li[class*='job']")           
        or soup.select("article[class*='job']")      
    )

    # Fallback: collect all <a> tags that link to a job detail page
    if not cards:
        job_links = soup.select("a[href*='/job/']") or soup.select("a[href*='/jobs/']") or soup.select("a[href*='jobId']") or soup.select("a[href*='jobseqno']")
        seen = set()
        for link in job_links:
            parent = link.parent
            if parent and id(parent) not in seen:
                cards.append(parent)
                seen.add(id(parent))

    for card in cards:
        # Title
        title_el = (
            card.select_one("[class*='jobTitle'] a")
            or card.select_one("[class*='title'] a")
            or card.select_one("h2 a") or card.select_one("h3 a")
            or card.select_one("a[href*='/job/']")
            or card.select_one("a[href*='jobId']")
            or card.select_one("a")
        )
        title = title_el.get_text(strip=True) if title_el else ""
        if not is_valid_job_title(title):
            continue

        # Job URL
        href = title_el.get("href", "") if title_el else ""
        job_url = href if href.startswith("http") else (BASE_URL + href if href else "")

        # Job ID — try URL param, data attribute, or last path segment
        job_id = ""
        if href:
            id_match = re.search(r'[?&](?:jobId|jobseqno|id)=([^&]+)', href)
            if id_match:
                job_id = id_match.group(1)
            else:
                job_id = href.rstrip("/").split("/")[-1]
        if not job_id:
            id_el = card.get("data-job-id") or card.get("data-id")
            job_id = id_el or str(abs(hash(title + job_url)))

        # Location
        loc_el = (
            card.select_one("[class*='jobLocation']") or card.select_one("[class*='location']")
            or card.select_one("[class*='city']") or card.select_one("[class*='country']")
        )
        location_text = loc_el.get_text(strip=True) if loc_el else "India"
        city = location_text.split(",")[0].strip() if location_text else "India"

        # Department / Business Unit
        dept_el = (
            card.select_one("[class*='department']") or card.select_one("[class*='category']")
            or card.select_one("[class*='function']") or card.select_one("[class*='jobType']")
        )
        department = dept_el.get_text(strip=True) if dept_el else ""

        # Date posted
        date_el = (
            card.select_one("[class*='date']") or card.select_one("[class*='posted']")
            or card.select_one("time")
        )
        raw_date = date_el.get_text(strip=True) if date_el else ""
        # Normalize "30 Mar 2026" -> "2026-03-30" or keep as-is
        date_posted = datetime.now().strftime("%Y-%m-%d")
        if raw_date:
            try:
                date_posted = datetime.strptime(raw_date, "%d %b %Y").strftime("%Y-%m-%d")
            except ValueError:
                try:
                    date_posted = datetime.strptime(raw_date, "%B %d, %Y").strftime("%Y-%m-%d")
                except ValueError:
                    pass  # keep today's date

        # JD text from card (full text; detail fetch done later)
        jd_text = card.get_text(" ", strip=True)

        jobs.append({
            "job_id": str(job_id),
            "title": title,
            "company_name": "Alstom",
            "job_url": job_url,
            "source_api_url": BASE_URL + SEARCH_PATH,
            "business_unit": department,
            "raw_jd_text": jd_text,
            "location_city": city,
            "location_country": "India",
            "industry": "Transportation / Railway Engineering",
            "date_posted": date_posted,
            "is_active": True,
            "salary_currency": "INR",
            "source_platform": "SAP SuccessFactors / Jobs2Web",
        })

    return jobs


def fetch_jd_detail(driver, url):
    """Visit a job detail page and extract the full JD text."""
    if not url:
        return ""
    try:
        driver.get(url)
        time.sleep(random.uniform(2, 4))
        soup = BeautifulSoup(driver.page_source, "lxml")
        for sel in [
            "[class*='jobDescription']", "[class*='job-description']",
            "[class*='jd-content']",     "[class*='description']",
            "[id*='description']",       "[id*='jobDescription']",
            "article",                   "main",
        ]:
            el = soup.select_one(sel)
            if el and len(el.get_text(strip=True)) > 100:
                return el.get_text(" ", strip=True)
        body = soup.select_one("body")
        return body.get_text(" ", strip=True)[:6000] if body else ""
    except Exception as e:
        print(f"    [WARN] JD detail fetch failed for {url}: {e}")
        return ""


# ── MAIN SCRAPE LOOP ──────────────────────────────────────────────────────────
alstom_jobs = []
seen_ids = set()
PAGE_SIZE = 25  # Jobs2Web uses 25 results per page
MAX_JOBS = 500

driver = setup_selenium()
try:
    startrow = 0
    consecutive_empty = 0

    while startrow < MAX_JOBS:
        url = build_search_url(COUNTRY_PARAM, startrow)
        print(f"  Fetching page startrow={startrow}: {url}")
        driver.get(url)

        # Wait for job results to render
        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR,
                    "tr.data-row, [class*='jobResultItem'], [class*='job-result'], "
                    "a[href*='/job/'], a[href*='jobId'], a[href*='jobseqno']"
                ))
            )
        except Exception:
            print(f"  [WARN] Timeout waiting for job cards at startrow={startrow}")
            time.sleep(5)  # extra wait before giving up on this page

        soup = BeautifulSoup(driver.page_source, "lxml")
        page_jobs = parse_job_cards(soup)

        # Deduplicate
        new_jobs = [j for j in page_jobs if j["job_id"] not in seen_ids]
        for j in new_jobs:
            seen_ids.add(j["job_id"])
        alstom_jobs.extend(new_jobs)

        print(f"    Got {len(new_jobs)} new jobs (total: {len(alstom_jobs)})")

        if not new_jobs:
            consecutive_empty += 1
            if consecutive_empty >= 2:
                print("  No new jobs on 2 consecutive pages — stopping pagination.")
                break
        else:
            consecutive_empty = 0

        # Check for a "next page" link as confirmation pagination is alive
        next_link = soup.select_one(
            "a[class*='next'], a[aria-label*='Next'], a[aria-label*='next'], "
            "[class*='pagination'] a[href*='startrow']"
        )
        if not next_link and len(page_jobs) < PAGE_SIZE:
            print("  Last page reached (no next link and partial page).")
            break

        startrow += PAGE_SIZE
        time.sleep(random.uniform(1.5, 3.0))

    # ── FETCH FULL JDs FOR TOP JOBS ───────────────────────────────────────────
    jobs_with_url = [j for j in alstom_jobs if j.get("job_url")]
    JD_FETCH_LIMIT = min(len(jobs_with_url), 100)  # cap to avoid very long runs
    if JD_FETCH_LIMIT:
        print(f"\n  Fetching full JD details for {JD_FETCH_LIMIT} jobs...")
        for i, job in enumerate(alstom_jobs[:JD_FETCH_LIMIT]):
            if job.get("raw_jd_text") and len(job["raw_jd_text"]) > 200:
                continue  # already has decent text
            jd = fetch_jd_detail(driver, job["job_url"])
            if jd:
                alstom_jobs[i]["raw_jd_text"] = jd
            if (i + 1) % 10 == 0:
                print(f"    Fetched {i + 1}/{JD_FETCH_LIMIT} JDs")

except Exception as e:
    print(f"  [ERROR] {e}")
    import traceback; traceback.print_exc()
finally:
    driver.quit()

print(f"\nTotal Alstom India jobs scraped: {len(alstom_jobs)}")

In [ ]:
df_alstom = save_results(alstom_jobs, "Alstom", OUTPUT_DIR)
if df_alstom is not None:
    print(f"\nSample jobs:")
    cols = ["title", "location_city", "seniority_level", "business_unit", "job_url"]
    cols = [c for c in cols if c in df_alstom.columns]
    print(df_alstom[cols].head(10).to_string())